# Практична 07. GPU і модель CUDA: Мандельброт на відеокарті

Та сама задача, що й у практичних 02–06: множина Мандельброта на ділянці [-2, 1] × [-1, 1],
`max_iter = 500`. Цього разу вона рахується на відеокарті Google Colab.

Що міряється: **три окремі числа** — копіювання на карту, ядро, копіювання назад. Перед кожним
зупиненням секундоміра стоїть `cuda.synchronize()`, бо запуск ядра не чекає його завершення.

**Умови замірів.** Усі часи — `time.perf_counter()`, мінімум із 3 прогонів. Перший запуск ядра
(компіляція під карту) у прогони не входить і наводиться окремо. Процесорний еталон у цьому зошиті —
моя функція `@njit(parallel=True)` із практичної 06, порахована на процесорі Colab (а не на моєму ПК).

## Крок 0. Карта і середовище

In [ ]:
!nvidia-smi

In [ ]:
import numba, numpy as np
from numba import cuda

print("numba", numba.__version__)
print("cuda.is_available:", cuda.is_available())
dev = cuda.get_current_device()
print("карта:", dev.name.decode() if isinstance(dev.name, bytes) else dev.name)
SM = dev.MULTIPROCESSOR_COUNT
print("мультипроцесорів (SM):", SM, "| розмір варпу:", dev.WARP_SIZE)

import os
print("ядер процесора в Colab:", os.cpu_count())

## Крок 1. Ядро: один потік — одна точка

Ядро виконує кожен потік. Два зовнішні цикли (по рядках і стовпцях) з практичної 02 зникають: їх замінює
сітка потоків. Внутрішній цикл — той самий, що в практиці 06, але на дійсних числах (`zx`, `zy`), бо ядро
CUDA працює з простою арифметикою. Координати точки рахуються тими самими формулами, що в `mandelbrot.py`
(`крок = (x_max - x_min) / w`, `cy = y_min + (y_max - y_min) * y / h`), інакше сітка не збіжиться з еталоном.

Потік, якому не дісталось точки (сітка блоків більша за картинку), нічого не робить: перевірка меж
`if x >= w or y >= h: return`.

In [ ]:
import math
import time
import numpy as np
from numba import cuda, njit, prange
import matplotlib.pyplot as plt

X_MIN, X_MAX, Y_MIN, Y_MAX = -2.0, 1.0, -1.0, 1.0
MAX_ITER = 500
W, H = 1200, 800


@cuda.jit
def mandelbrot_gpu(out, x_min, x_max, y_min, y_max, max_iter):
    x, y = cuda.grid(2)                              # хто я в сітці
    h, w = out.shape
    if x >= w or y >= h:                             # потоку не дісталось точки
        return
    krok = (x_max - x_min) / w
    cx = x_min + krok * x
    cy = y_min + (y_max - y_min) * y / h
    zx = 0.0
    zy = 0.0
    n = max_iter
    for k in range(max_iter):
        zx, zy = zx * zx - zy * zy + cx, 2.0 * zx * zy + cy
        if zx * zx + zy * zy > 4.0:
            n = k                                    # як в еталоні: номер кроку втечі
            break
    out[y, x] = n


def kilkist_blokiv(w, h, potokiv):
    # стеля від ділення: блоків має вистачити на ВСІ точки, а не лише на кратні розміру блока
    return ((w + potokiv[0] - 1) // potokiv[0], (h + potokiv[1] - 1) // potokiv[1])


potokiv = (16, 16)                                   # 256 потоків у блоці
blokiv = kilkist_blokiv(W, H, potokiv)
print("1200×800, блок 16×16 -> блоків:", blokiv, "=", blokiv[0] * blokiv[1])
print("1000×700 (не ділиться на 16) -> блоків:", kilkist_blokiv(1000, 700, potokiv), "(має бути (63, 44))")

d_out = cuda.device_array((H, W), dtype=np.int32)
mandelbrot_gpu[blokiv, potokiv](d_out, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)
out = d_out.copy_to_host()
print("мінімум і максимум у результаті:", out.min(), out.max(), "(має бути від 0 до", MAX_ITER, ")")

plt.figure(figsize=(9, 6))
plt.imshow(out, cmap="magma_r", extent=(X_MIN, X_MAX, Y_MAX, Y_MIN))
plt.title("Мандельброт, GPU, %d×%d" % (W, H))
plt.show()

## Крок 2. Еталон у тому самому зошиті

Порівнювати результат треба з тим, що порахувала ця сама машина (процесор Colab), а не з моїм ПК.
Беру свою функцію `@njit(parallel=True)` із `4-numba/numba_versiya.py` дослівно: комплексне `z`,
`abs(z) > 2.0`, ті самі формули координат.

In [ ]:
@njit(parallel=True)
def mandelbrot_prange(w, h, max_iter, x_min, x_max, y_min, y_max):
    out = np.empty((h, w), dtype=np.int32)
    krok = (x_max - x_min) / w
    for y in prange(h):
        im = y_min + (y_max - y_min) * y / h
        for x in range(w):
            c = complex(x_min + krok * x, im)
            z = 0j
            n = max_iter
            for k in range(max_iter):
                z = z * z + c
                if abs(z) > 2.0:
                    n = k
                    break
            out[y, x] = n
    return out


t0 = time.perf_counter()
mandelbrot_prange(2, 2, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX)       # компіляція окремо
print("компіляція prange на процесорі Colab: %.2f c" % (time.perf_counter() - t0))

et = mandelbrot_prange(W, H, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX)
rozbizhnyh = int((out != et).sum())
print("розбіжних точок GPU і процесорного еталона:", rozbizhnyh, "з", W * H)

if rozbizhnyh:
    ys, xs = np.nonzero(out != et)
    print("перші 8: (рядок, стовпець, еталон, GPU)")
    for y, x in list(zip(ys, xs))[:8]:
        print("  ", y, x, et[y, x], out[y, x])

## Крок 3. Три числа: туди, ядро, назад

Запуск ядра повертає керування одразу. Без `cuda.synchronize()` секундомір зупиниться, поки ядро
ще працює, і виміряє лише запуск. Тому синхронізація стоїть перед зупинкою кожного таймера.

Ядро нічого не читає зі вхідних даних, тому «копіювання туди» я міряю на буфері того самого розміру,
що й результат (`cuda.to_device` порожнього масиву `H×W` типу `int32`). Це вартість шини для задачі,
яка б мала такий вхід.

Запуск у цій функції не змінює порядок: туди → синхронізація → ядро → синхронізація → назад → синхронізація.
`to_device` і `copy_to_host` з пам'яті, що не закріплена, самі чекають закінчення копіювання,
а от запуск ядра — ні, тому для ядра `synchronize` обов'язковий.

In [ ]:
def zamir_gpu(w, h, blok=(16, 16), n=3, kernel=None, **kw):
    kernel = kernel or mandelbrot_gpu
    host = np.empty((h, w), np.int32)
    blokiv = kilkist_blokiv(w, h, blok)
    d = cuda.device_array((h, w), np.int32)
    kernel[blokiv, blok](d, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)     # розігрів: компіляція під карту
    cuda.synchronize()

    prohony = []
    for _ in range(n):
        cuda.synchronize()
        t0 = time.perf_counter()
        d = cuda.to_device(host)                                       # 1) туди
        cuda.synchronize()
        t1 = time.perf_counter()
        kernel[blokiv, blok](d, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)  # 2) ядро
        cuda.synchronize()
        t2 = time.perf_counter()
        res = d.copy_to_host()                                         # 3) назад
        cuda.synchronize()
        t3 = time.perf_counter()
        prohony.append((t1 - t0, t2 - t1, t3 - t2, res))
    best = min(prohony, key=lambda p: p[0] + p[1] + p[2])
    return best


# перший запуск ядра окремо (компіляція)
cuda.synchronize()
t0 = time.perf_counter()
d_tmp = cuda.device_array((16, 16), np.int32)
mandelbrot_gpu[(1, 1), (16, 16)](d_tmp, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)
cuda.synchronize()
print("перший запуск ядра (компіляція) на 16×16: %.3f c" % (time.perf_counter() - t0))

t_tudy, t_yadro, t_nazad, res = zamir_gpu(W, H)
razom = t_tudy + t_yadro + t_nazad
print("1200×800, блоки 16×16, мінімум із 3:")
print("  туди  : %.3f мс" % (t_tudy * 1e3))
print("  ядро  : %.3f мс" % (t_yadro * 1e3))
print("  назад : %.3f мс" % (t_nazad * 1e3))
print("  частка копіювання: %.1f %%" % (100 * (t_tudy + t_nazad) / razom))
print("розбіжних точок з еталоном:", int((res != et).sum()))

### Пастка: ядро без `synchronize` і з ним

In [ ]:
d_p = cuda.device_array((H, W), np.int32)
blokiv = kilkist_blokiv(W, H, (16, 16))
cuda.synchronize()

t0 = time.perf_counter()
mandelbrot_gpu[blokiv, (16, 16)](d_p, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)
bez = time.perf_counter() - t0                       # лише запуск: керування повернулось одразу
cuda.synchronize()

t0 = time.perf_counter()
mandelbrot_gpu[blokiv, (16, 16)](d_p, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)
cuda.synchronize()
z = time.perf_counter() - t0

print("без synchronize: %.3f мс" % (bez * 1e3))
print("з synchronize:   %.3f мс" % (z * 1e3))
print("різниця: у %.0f разів — це і є «прискорення», якого немає" % (z / bez))

## Крок 4. Три розміри задачі

Для кожного розміру: три складові, частка копіювання і, для порівняння, той самий розмір на процесорі Colab
(`@njit(parallel=True)`, мінімум із 3 прогонів, без компіляції).

In [ ]:
def min_cpu(w, h, n=3):
    kr = 1e9
    for _ in range(n):
        t0 = time.perf_counter()
        mandelbrot_prange(w, h, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX)
        kr = min(kr, time.perf_counter() - t0)
    return kr


rozmiry = [(300, 200), (1200, 800), (3000, 2000)]
rjadky = []
print("| Розмір | Туди, мс | Ядро, мс | Назад, мс | Частка копіювання | GPU разом, мс | CPU prange (Colab), мс |")
print("|---|---|---|---|---|---|---|")
for w, h in rozmiry:
    a, b, c, _ = zamir_gpu(w, h)
    cpu = min_cpu(w, h)
    rjadky.append((w, h, a, b, c, cpu))
    print("| %d×%d | %.3f | %.3f | %.3f | %.1f %% | %.3f | %.3f |"
          % (w, h, a * 1e3, b * 1e3, c * 1e3, 100 * (a + c) / (a + b + c), (a + b + c) * 1e3, cpu * 1e3))

In [ ]:
# скільки разів GPU швидший за процесор Colab: тільки ядро і разом із дорогою
for w, h, a, b, c, cpu in rjadky:
    print("%d×%d:  CPU/GPU-ядро = ×%.2f,  CPU/GPU-разом = ×%.2f" % (w, h, cpu / b, cpu / (a + b + c)))

## Крок 7. Потоків менше, ніж точок: цикл із кроком сітки

Сітка фіксованого розміру, не залежить від `W` і `H`: блоки 16×16, блоків `(SM × k, 4)`.
Кожен потік починає з точки `(x, y)`, що відповідає його місцю в сітці, і йде по рядках і стовпцях
з кроком, що дорівнює розміру всієї сітки (`cuda.gridsize(2)`): так усі точки покриті й жодна не рахується двічі.

In [ ]:
@cuda.jit
def mandelbrot_stride(out, x_min, x_max, y_min, y_max, max_iter):
    h, w = out.shape
    start_x, start_y = cuda.grid(2)
    krok_x, krok_y = cuda.gridsize(2)
    krok = (x_max - x_min) / w
    for y in range(start_y, h, krok_y):
        cy = y_min + (y_max - y_min) * y / h
        for x in range(start_x, w, krok_x):
            cx = x_min + krok * x
            zx = 0.0
            zy = 0.0
            n = max_iter
            for k in range(max_iter):
                zx, zy = zx * zx - zy * zy + cx, 2.0 * zx * zy + cy
                if zx * zx + zy * zy > 4.0:
                    n = k
                    break
            out[y, x] = n


def zamir_stride(w, h, k, n=3):
    blok = (16, 16)
    blokiv = (SM * k, 4)
    d = cuda.device_array((h, w), np.int32)
    mandelbrot_stride[blokiv, blok](d, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)   # розігрів
    cuda.synchronize()
    kr = 1e9
    for _ in range(n):
        cuda.synchronize()
        t0 = time.perf_counter()
        mandelbrot_stride[blokiv, blok](d, X_MIN, X_MAX, Y_MIN, Y_MAX, MAX_ITER)
        cuda.synchronize()
        kr = min(kr, time.perf_counter() - t0)
    return kr, d.copy_to_host(), blokiv[0] * blokiv[1] * 256


print("потік на точку (крок 1), 1200×800: %.3f мс" % (t_yadro * 1e3))
print()
print("| k | Блоків | Потоків | Точок на потік | Ядро, мс | Розбіжних з GPU-ядром кроку 1 | Розбіжних з CPU-еталоном |")
print("|---|---|---|---|---|---|---|")
for k in (1, 2, 4, 8, 16):
    t, r, potokiv_vsogo = zamir_stride(W, H, k)
    print("| %d | %d×4 | %d | %.1f | %.3f | %d | %d |"
          % (k, SM * k, potokiv_vsogo, W * H / potokiv_vsogo, t * 1e3,
             int((r != res).sum()), int((r != et).sum())))

## ⭐ Розмір блока і пропускна здатність шини

In [ ]:
print("| Блок | Потоків у блоці | Варпів | Ядро, мс |")
print("|---|---|---|---|")
for blok in ((8, 8), (16, 16), (32, 32), (10, 10)):
    _, t_y, _, r = zamir_gpu(W, H, blok=blok)
    potokiv_u_bloci = blok[0] * blok[1]
    print("| %d×%d | %d | %.3f | %.3f |" % (blok[0], blok[1], potokiv_u_bloci, potokiv_u_bloci / 32, t_y * 1e3))
    assert int((r != res).sum()) == 0, "результат залежить від розміру блока"

# 10×10: у кожного блока 100 потоків = 3 повних варпи + 4 потоки в четвертому
potokiv_u_bloci = 100
varpiv = -(-potokiv_u_bloci // 32)
print()
print("10×10: %d варпи, у останньому активні 4 потоки з 32; даремні слоти: %d з %d (%.1f %%)"
      % (varpiv, varpiv * 32 - potokiv_u_bloci, varpiv * 32, 100 * (varpiv * 32 - potokiv_u_bloci) / (varpiv * 32)))

bajt = W * H * 4
print()
print("Масив 1200×800 int32: %.2f МіБ" % (bajt / 2**20))
print("Копіювання туди: %.2f ГБ/с, назад: %.2f ГБ/с (PCIe 3.0 ×16 теоретично 15.75 ГБ/с в один бік)"
      % (bajt / t_tudy / 1e9, bajt / t_nazad / 1e9))